# Full-text Model with Triplet Loss

This notebook will train the detection model at the full-text embedding data for the original non-paraphrased texts. The inputs should be three embedding files from the same embedding model.

# Set up

In [1]:
import numpy as np
import pandas as pd

## Class definition

Define the classes for our model. You do not need to change this section.

In [2]:
import tensorflow as tf

class FeedForward(tf.keras.layers.Layer):
  def __init__(self, d_model, dropout_rate=0.1):
    super().__init__()
    self.seq = tf.keras.Sequential([
      tf.keras.layers.Dense(d_model, activation='relu'),
      tf.keras.layers.Dense(d_model),
      tf.keras.layers.Dropout(dropout_rate)
    ])
    self.add = tf.keras.layers.Add()
    self.layer_norm = tf.keras.layers.LayerNormalization()

  def call(self, x):
    x = self.add([x, self.seq(x)])
    x = self.layer_norm(x)
    return x


class FinetuneEncoder(tf.keras.layers.Layer):
  def __init__(self, ff_layers, dropout_rate=0.1):
    super().__init__()
    self.num_blocks = ff_layers
    self.ff_layers = [FeedForward(d_model, dropout_rate) for d_model in self.num_blocks]
    self.embedding = tf.keras.layers.Lambda(lambda x: tf.math.l2_normalize(x, axis=1))

  def call(self, x):
    for layer in self.ff_layers:
      x = layer(x)
    x = self.embedding(x)
    return x


class DistanceLayer(tf.keras.layers.Layer):
  def __init__(self, **kwargs):
      super().__init__(**kwargs)

  def call(self, anchor, positive, negative):
      ap_distance = tf.reduce_sum(tf.square(anchor - positive), -1)
      an_distance = tf.reduce_sum(tf.square(anchor - negative), -1)
      return (ap_distance, an_distance)


class TripletModel(tf.keras.Model):
  def __init__(self, siamese_network, margin=0.5):
      super().__init__()
      self.siamese_network = siamese_network
      self.margin = margin
      self.loss_tracker = tf.keras.metrics.Mean(name="loss")

  def call(self, inputs):
      return self.siamese_network(inputs)

  def train_step(self, data):
      with tf.GradientTape() as tape:
          loss = self._compute_loss(data)
      gradients = tape.gradient(loss, self.siamese_network.trainable_weights)
      self.optimizer.apply_gradients(
          zip(gradients, self.siamese_network.trainable_weights)
      )
      self.loss_tracker.update_state(loss)
      return {"loss": self.loss_tracker.result()}

  def test_step(self, data):
      loss = self._compute_loss(data)
      self.loss_tracker.update_state(loss)
      return {"loss": self.loss_tracker.result()}

  def _compute_loss(self, data):
      ap_distance, an_distance = self.siamese_network(data)
      loss = ap_distance - an_distance
      loss = tf.maximum(loss + self.margin, 0.0)
      return loss

  @property
  def metrics(self):
      # We need to list our metrics here so the `reset_states()` can be
      # called automatically.
      return [self.loss_tracker]


from sklearn.metrics import f1_score

def f1(x,y,thres):
    yx = np.zeros(x.shape[0])
    yy = np.ones(y.shape[0])
    yhx = np.zeros(x.shape[0])
    yhx[x > thres] = 1
    yhy = np.zeros(y.shape[0])
    yhy[y > thres] = 1
    return f1_score(np.concatenate([yx,yy]), np.concatenate([yhx,yhy]))

# Function to build model and evaluate

In [3]:
def to_numeric_matrix(x):
    if isinstance(x, pd.Series):
        x = np.stack(x.to_list())
    else:
        x = np.asarray(x)

    return x.astype(np.float32)

answers_np = to_numeric_matrix(answers)
gpt1_np = to_numeric_matrix(gpt1)
gpt2_np = to_numeric_matrix(gpt2)

print(type(answers_np), answers_np.shape, answers_np.dtype)
print(type(gpt1_np), gpt1_np.shape, gpt1_np.dtype)
print(type(gpt2_np), gpt2_np.shape, gpt2_np.dtype)

NameError: name 'answers' is not defined

In [ ]:
def run_model(input_shape, n_layers, answers, gpt1, gpt2, epochs=3, seeds=(1111,2222,3333,4444,5555,6666,7777,8888,9999,1010)):
  valid_accuracies = []
  valid_f1s = []
  test_accuracies = []
  test_f1s = []

  for seed in seeds:
    np.random.seed(seed)

    #train-test-validation split
    sampling_indices = np.random.uniform(0,1,answers.shape[0])
    train_answers = answers[sampling_indices < 0.8]
    train_gpt1 = gpt1[sampling_indices < 0.8]
    train_gpt2 = gpt2[sampling_indices < 0.8]

    valid_answers = answers[(0.8 <= sampling_indices) & (sampling_indices < 0.9)]
    valid_gpt1 = gpt1[(0.8 <= sampling_indices) & (sampling_indices < 0.9)]
    valid_gpt2 = gpt2[(0.8 <= sampling_indices) & (sampling_indices < 0.9)]

    test_answers = answers[sampling_indices >= 0.9]
    test_gpt1 = gpt1[sampling_indices >= 0.9]
    test_gpt2 = gpt2[sampling_indices >= 0.9]

    #build model
    emb_shape = input_shape
    embedding = FinetuneEncoder([emb_shape]*n_layers)
    anchor_input = tf.keras.layers.Input(name="anchor", shape=[emb_shape])
    positive_input = tf.keras.layers.Input(name="positive", shape=[emb_shape])
    negative_input = tf.keras.layers.Input(name="negative", shape=[emb_shape])
    distances = DistanceLayer()(
        embedding(anchor_input),
        embedding(positive_input),
        embedding(negative_input),
    )
    distance_network = tf.keras.Model(
        inputs=[anchor_input, positive_input, negative_input], outputs=distances
    )
    triplet_model = TripletModel(distance_network, margin=0.1)

    #training
    triplet_model.compile(optimizer=tf.keras.optimizers.Adam(0.00001), weighted_metrics=[])
    triplet_model.fit([train_gpt1, train_gpt2, train_answers],
                      epochs=epochs, batch_size=2048,
                      validation_data=[valid_gpt1, valid_gpt2, valid_answers])

    #valid accuracy
    valid_emb_answers = embedding(valid_answers)
    valid_emb_gpt1 = embedding(valid_gpt1)
    valid_emb_gpt2 = embedding(valid_gpt2)
    valid_gpt_distances = ((valid_emb_gpt1 - valid_emb_gpt2)**2).numpy().sum(axis=1)
    valid_answer_distances = ((valid_emb_gpt1 - valid_emb_answers)**2).numpy().sum(axis=1)
    valid_accuracies.append((valid_gpt_distances < valid_answer_distances).mean())

    #tune threshold
    thresholds = []
    perf = []
    for thres in np.arange(0,1,0.01):
      thresholds.append(thres)
      perf.append(f1(valid_gpt_distances, valid_answer_distances, thres))

    triplet_thres = thresholds[perf.index(max(perf))]

    #valid f1
    valid_f1s.append(max(perf))

    #test accuracy
    test_emb_answers = embedding(test_answers)
    test_emb_gpt1 = embedding(test_gpt1)
    test_emb_gpt2 = embedding(test_gpt2)
    test_gpt_distances = ((test_emb_gpt1 - test_emb_gpt2)**2).numpy().sum(axis=1)
    test_answer_distances = ((test_emb_gpt1 - test_emb_answers)**2).numpy().sum(axis=1)
    test_accuracies.append((test_gpt_distances < test_answer_distances).mean())

    #test f1
    test_f1s.append(f1(test_gpt_distances, test_answer_distances, triplet_thres))
  return valid_accuracies, test_accuracies, valid_f1s, test_f1s

## Load Data

We need to load the embeddings for `answer`, `gpt1`, and `gpt2`

In [6]:
dataset = 'SQUAD_GPT4'

import pickle

with open(f'/home/aanis/Metric-Models-for-Detection-of-LLM-Texts/Data/{dataset}_answer_embs.obj', 'rb') as f:
  answers = pickle.load(f)
with open(f'/home/aanis/Metric-Models-for-Detection-of-LLM-Texts/Data/{dataset}_gpt1_embs.obj', 'rb') as f:
  gpt1 = pickle.load(f)
with open(f'/home/aanis/Metric-Models-for-Detection-of-LLM-Texts/Data/{dataset}_gpt2_embs.obj', 'rb') as f:
  gpt2 = pickle.load(f)

# Run Model

In [10]:
input_shape = 768
n_layers = 3
answers = np.asarray(np.stack(answers.to_list()), dtype=np.float32)
gpt1 = np.asarray(np.stack(gpt1.to_list()), dtype=np.float32)
gpt2 = np.asarray(np.stack(gpt2.to_list()), dtype=np.float32)

#run function to train and evaluate model
valid_accuracies, test_accuracies, valid_f1s, test_f1s = run_model(input_shape, n_layers, answers, gpt1, gpt2)

Epoch 1/50


/home/aanis/venv/lib/python3.11/site-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['anchor', 'positive', 'negative']
Received: inputs=(('Tensor(shape=(None, 768))', 'Tensor(shape=(None, 768))', 'Tensor(shape=(None, 768))'),)
  warnings.warn(msg)


2/2 ━━━━━━━━━━━━━━━━━━━━ 5s 1s/step - loss: 0.0069 - val_loss: 0.0052
Epoch 2/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 4s 848ms/step - loss: 0.0067 - val_loss: 0.0052
Epoch 3/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 448ms/step - loss: 0.0064 - val_loss: 0.0052
Epoch 4/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 458ms/step - loss: 0.0062 - val_loss: 0.0052
Epoch 5/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 533ms/step - loss: 0.0060 - val_loss: 0.0052
Epoch 6/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 414ms/step - loss: 0.0058 - val_loss: 0.0052
Epoch 7/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 369ms/step - loss: 0.0056 - val_loss: 0.0052
Epoch 8/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 492ms/step - loss: 0.0054 - val_loss: 0.0052
Epoch 9/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 359ms/step - loss: 0.0052 - val_loss: 0.0051
Epoch 10/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 356ms/step - loss: 0.0050 - val_loss: 0.0051
Epoch 11/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 990ms/step - loss: 0.0048 - val_loss: 0.0051
Epoch 12/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 374ms/step - loss: 0.0047 - val_loss: 0.0051
Epo

/home/aanis/venv/lib/python3.11/site-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['anchor', 'positive', 'negative']
Received: inputs=(('Tensor(shape=(None, 768))', 'Tensor(shape=(None, 768))', 'Tensor(shape=(None, 768))'),)
  warnings.warn(msg)


2/2 ━━━━━━━━━━━━━━━━━━━━ 4s 681ms/step - loss: 0.0074 - val_loss: 0.0064
Epoch 2/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 4s 3s/step - loss: 0.0072 - val_loss: 0.0064
Epoch 3/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 597ms/step - loss: 0.0069 - val_loss: 0.0064
Epoch 4/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 365ms/step - loss: 0.0067 - val_loss: 0.0064
Epoch 5/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 370ms/step - loss: 0.0065 - val_loss: 0.0063
Epoch 6/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 393ms/step - loss: 0.0062 - val_loss: 0.0063
Epoch 7/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 372ms/step - loss: 0.0060 - val_loss: 0.0063
Epoch 8/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 359ms/step - loss: 0.0058 - val_loss: 0.0063
Epoch 9/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 407ms/step - loss: 0.0056 - val_loss: 0.0063
Epoch 10/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 360ms/step - loss: 0.0054 - val_loss: 0.0063
Epoch 11/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 435ms/step - loss: 0.0053 - val_loss: 0.0062
Epoch 12/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 369ms/step - loss: 0.0051 - val_loss: 0.0062
Epo

...

2/2 ━━━━━━━━━━━━━━━━━━━━ 4s 912ms/step - loss: 0.0073 - val_loss: 0.0050
Epoch 2/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 869ms/step - loss: 0.0070 - val_loss: 0.0050
Epoch 3/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 626ms/step - loss: 0.0067 - val_loss: 0.0050
Epoch 4/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 436ms/step - loss: 0.0065 - val_loss: 0.0050
Epoch 5/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 983ms/step - loss: 0.0063 - val_loss: 0.0049
Epoch 6/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 616ms/step - loss: 0.0061 - val_loss: 0.0049
Epoch 7/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 393ms/step - loss: 0.0059 - val_loss: 0.0049
Epoch 8/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 614ms/step - loss: 0.0057 - val_loss: 0.0049
Epoch 9/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 533ms/step - loss: 0.0055 - val_loss: 0.0049
Epoch 10/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 680ms/step - loss: 0.0053 - val_loss: 0.0049
Epoch 11/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 416ms/step - loss: 0.0051 - val_loss: 0.0049
Epoch 12/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 358ms/step - loss: 0.0049 - val_loss: 0.0049


/home/aanis/venv/lib/python3.11/site-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['anchor', 'positive', 'negative']
Received: inputs=(('Tensor(shape=(None, 768))', 'Tensor(shape=(None, 768))', 'Tensor(shape=(None, 768))'),)
  warnings.warn(msg)


2/2 ━━━━━━━━━━━━━━━━━━━━ 5s 1s/step - loss: 0.0072 - val_loss: 0.0058
Epoch 2/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 583ms/step - loss: 0.0069 - val_loss: 0.0058
Epoch 3/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 406ms/step - loss: 0.0067 - val_loss: 0.0058
Epoch 4/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 362ms/step - loss: 0.0064 - val_loss: 0.0058
Epoch 5/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 980ms/step - loss: 0.0062 - val_loss: 0.0058
Epoch 6/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 576ms/step - loss: 0.0060 - val_loss: 0.0057
Epoch 7/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 1s/step - loss: 0.0058 - val_loss: 0.0057
Epoch 8/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 381ms/step - loss: 0.0056 - val_loss: 0.0057
Epoch 9/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 477ms/step - loss: 0.0054 - val_loss: 0.0057
Epoch 10/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 368ms/step - loss: 0.0052 - val_loss: 0.0057
Epoch 11/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 826ms/step - loss: 0.0050 - val_loss: 0.0057
Epoch 12/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 615ms/step - loss: 0.0048 - val_loss: 0.0057
Epoch 

/home/aanis/venv/lib/python3.11/site-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['anchor', 'positive', 'negative']
Received: inputs=(('Tensor(shape=(None, 768))', 'Tensor(shape=(None, 768))', 'Tensor(shape=(None, 768))'),)
  warnings.warn(msg)


2/2 ━━━━━━━━━━━━━━━━━━━━ 4s 656ms/step - loss: 0.0075 - val_loss: 0.0043
Epoch 2/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 405ms/step - loss: 0.0072 - val_loss: 0.0043
Epoch 3/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 417ms/step - loss: 0.0070 - val_loss: 0.0042
Epoch 4/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 491ms/step - loss: 0.0068 - val_loss: 0.0042
Epoch 5/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 566ms/step - loss: 0.0065 - val_loss: 0.0042
Epoch 6/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 405ms/step - loss: 0.0063 - val_loss: 0.0042
Epoch 7/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 423ms/step - loss: 0.0061 - val_loss: 0.0042
Epoch 8/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 357ms/step - loss: 0.0059 - val_loss: 0.0042
Epoch 9/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 390ms/step - loss: 0.0057 - val_loss: 0.0042
Epoch 10/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 503ms/step - loss: 0.0055 - val_loss: 0.0042
Epoch 11/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 528ms/step - loss: 0.0053 - val_loss: 0.0042
Epoch 12/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 526ms/step - loss: 0.0051 - val_loss: 0.0041


In [11]:
np.mean(valid_accuracies), np.mean(test_accuracies), np.mean(valid_f1s), np.mean(test_f1s)

(np.float64(0.9824856263035151),
 np.float64(0.9799606470222428),
 np.float64(0.8853240077328719),
 np.float64(0.8769631822183704))